# 03 — Las 30 transforms que usás el 95% del tiempo

Pasantía rápida por las operaciones más comunes. Después de esto, sabés
leer cualquier `df.???` de StackOverflow sin asustarte.

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("03_transforms_comunes")
    .master("spark://spark-master:7077")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

df = spark.read.parquet("/home/jovyan/work/data/sample/ventas/ventas_parquet")

## 1. Select, filter, where

`filter` y `where` son sinónimos. Usá el que prefieras.

In [ ]:
df.select("ciudad", "producto", "precio").show(5)
df.filter(F.col("precio") > 40000).show(5)
df.where((F.col("ciudad") == "Bogotá") & (F.col("producto") == "A")).show(5)

## 2. withColumn: agregar o modificar columnas

In [ ]:
df2 = (
    df.withColumn("ingreso", F.col("cantidad") * F.col("precio"))
      .withColumn("anio", F.year("fecha"))
      .withColumn("mes", F.month("fecha"))
      .withColumn("ciudad_upper", F.upper("ciudad"))
)
df2.show(5)

## 3. withColumnRenamed: renombrar columnas

Útil después de un join que duplicó nombres.

In [ ]:
df2.select(F.col("id").alias("venta_id"), "ciudad").show(3)
df2.withColumnRenamed("ciudad_upper", "ciudad_mayus").show(3)

## 4. drop: eliminar columnas

In [ ]:
df.drop("anio", "mes").columns

## 5. distinct y dropDuplicates

In [ ]:
df.select("ciudad").distinct().show()
df.dropDuplicates(["ciudad", "producto"]).show(5)

## 6. orderBy / sort

In [ ]:
df.orderBy(F.col("precio").desc()).show(5)
df.orderBy("ciudad", F.col("precio").desc()).show(5)

## 7. groupBy / agg

In [ ]:
df.groupBy("ciudad", "producto").agg(
    F.count("*").alias("n_ventas"),
    F.sum("cantidad").alias("total_unidades"),
    F.round(F.avg("precio"), 2).alias("precio_promedio"),
    F.min("precio").alias("min"),
    F.max("precio").alias("max"),
    F.expr("percentile_approx(precio, 0.95)").alias("p95"),
).orderBy("ciudad", "producto").show(10, truncate=False)

## 8. when / otherwise: condicionales encadenadas

El `CASE WHEN ... THEN ... ELSE ... END` de SQL.

In [ ]:
df.withColumn(
    "rango_precio",
    F.when(F.col("precio") < 10000, "bajo")
     .when(F.col("precio") < 30000, "medio")
     .when(F.col("precio") < 45000, "alto")
     .otherwise("premium"),
).groupBy("rango_precio").count().orderBy("rango_precio").show()

## 9. Joins

Spark tiene 5 estrategias. La más común: `inner` y `left`.

**Truco**: si la tabla derecha entra en memoria, hacé `broadcast`:
`df.join(broadcast(ciudades_df), "ciudad")`.

In [ ]:
pob = spark.createDataFrame(
    [
        ("Bogotá", 8000000),
        ("Medellín", 2500000),
        ("Cali", 2200000),
        ("Barranquilla", 1200000),
        ("Cartagena", 900000),
    ],
    ["ciudad", "poblacion"],
)

joined = df.join(F.broadcast(pob), "ciudad")
joined.groupBy("ciudad", "poblacion").agg(
    F.round(F.avg("precio"), 2).alias("precio_promedio")
).orderBy(F.col("poblacion").desc()).show()

## 10. Window functions

Ranking, running totals, lag/lead. La navaja suiza del análisis.

In [ ]:
w = Window.partitionBy("ciudad").orderBy(F.col("precio").desc())

df.withColumn("rank_en_ciudad", F.row_number().over(w)) \
  .filter(F.col("rank_en_ciudad") <= 3) \
  .select("ciudad", "producto", "precio", "rank_en_ciudad") \
  .orderBy("ciudad", "rank_en_ciudad") \
  .show(15)

In [ ]:
# Running total por ciudad
w_running = Window.partitionBy("ciudad").orderBy("fecha").rowsBetween(
    Window.unboundedPreceding, Window.currentRow
)
df.withColumn("running_total", F.sum("cantidad").over(w_running)) \
  .filter(F.col("ciudad") == "Bogotá") \
  .select("fecha", "cantidad", "running_total") \
  .orderBy("fecha") \
  .show(10)

## 11. Strings, fechas, nulls — los clásicos

In [ ]:
# Strings
df.select(
    F.lower("ciudad").alias("ciudad_lower"),
    F.concat_ws(" - ", "ciudad", "producto").alias("concat"),
    F.substring("ciudad", 1, 3).alias("primeros_3"),
    F.length("ciudad").alias("largo"),
).show(3)

# Fechas
df.select(
    F.current_date().alias("hoy"),
    F.datediff(F.current_date(), F.col("fecha")).alias("dias_desde"),
    F.date_add(F.col("fecha"), 7).alias("fecha_mas_7"),
    F.date_trunc("month", F.col("fecha")).alias("inicio_mes"),
).show(3)

# Nulls
df.select(
    F.coalesce(F.col("precio"), F.lit(0)).alias("precio_safe"),
    F.when(F.col("precio").isNull(), "falta").otherwise("ok").alias("estado"),
).show(3)

## ✅ Checkpoint

- [x] Manejo select, filter, withColumn, groupBy, orderBy.
- [x] Sé hacer joins y conozco `broadcast`.
- [x] Sé usar window functions para ranking y running totals.
- [x] Sé tratar strings, fechas y nulls.

**Siguiente**: `04_spark_sql.ipynb` — la misma DataFrame API pero en
SQL puro, ideal para SQL-first teams.